# 01 - Analisis Exploratorio de Datos

## 1. Contexto del problema
Este notebook corresponde al caso **Logistica 4.0: Prediccion de Tiempos de Entrega (Last Mile)**.

Objetivo de esta etapa:
- Explorar la calidad del dataset.
- Detectar problemas de limpieza y consistencia.
- Analizar el comportamiento de la variable objetivo `target_tiempo_entrega`.
- Preparar datos y funciones reutilizables para modelado posterior.

Este notebook **solo realiza EDA y limpieza exploratoria**. No entrena modelos finales ni ejecuta optimizacion de hiperparametros.


## 2. Importacion de librerias y modulos


In [ ]:
from pathlib import Path
import shutil
import sys
import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

for relative_dir in [
    "data/raw",
    "data/processed",
    "results/metrics",
    "results/plots",
    "results/reports",
    "docs",
]:
    (PROJECT_ROOT / relative_dir).mkdir(parents=True, exist_ok=True)

from src import data_loading, data_preprocessing, exploratory_analysis, feature_engineering, visualization

importlib.reload(data_loading)
importlib.reload(data_preprocessing)
importlib.reload(exploratory_analysis)
importlib.reload(feature_engineering)
importlib.reload(visualization)

print(f"Project root: {PROJECT_ROOT}")


## 3. Carga del dataset


In [ ]:
def resolve_dataset_path() -> Path:
    candidates = [
        PROJECT_ROOT / "data" / "raw" / "5_logistica_40.csv",
        Path.home() / "Descargas" / "5_logistica_40.csv",
        Path.home() / "Downloads" / "5_logistica_40.csv",
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        "No se encontro 5_logistica_40.csv en data/raw ni en Descargas/Downloads. "
        f"Rutas revisadas: {candidates}"
    )


dataset_path = resolve_dataset_path()
project_raw_path = PROJECT_ROOT / "data" / "raw" / "5_logistica_40.csv"

if dataset_path != project_raw_path and not project_raw_path.exists():
    shutil.copy2(dataset_path, project_raw_path)
    dataset_path = project_raw_path

print(f"Ruta dataset usada: {dataset_path}")
df = data_loading.load_dataset(str(dataset_path))

print("Shape:", df.shape)
print("Columnas:", df.columns.tolist())
display(df.head())
display(df.tail())
df.info()


## 4. Vista general del dataset


In [ ]:
overview_table = exploratory_analysis.dataset_overview(df)
column_types_table = exploratory_analysis.column_types_summary(df)

display(overview_table)
display(column_types_table)


## 5. Descripcion de columnas y rol dentro del proyecto


In [ ]:
column_roles = pd.DataFrame(
    [
        ["distancia_km", "Numerica", "Distancia recorrida en km", "Feature numerica"],
        ["trafico_nivel", "Categorica/Ordinal", "Nivel de trafico en la ruta", "Feature categorica/ordinal"],
        ["clima", "Categorica", "Condicion climatica del despacho", "Feature categorica"],
        ["tipo_vehiculo", "Categorica", "Tipo de vehiculo de reparto", "Feature categorica"],
        ["peso_carga_kg", "Numerica", "Peso de la carga en kilogramos", "Feature numerica"],
        ["paradas_previas", "Numerica discreta", "Cantidad de paradas antes de la entrega", "Feature numerica discreta"],
        ["experiencia_chofer_anios", "Numerica discreta", "Experiencia del chofer en anios", "Feature numerica discreta"],
        ["hora_despacho", "Temporal", "Hora de salida del despacho", "Feature temporal"],
        ["costo_envio", "Numerica", "Costo del envio", "Feature numerica"],
        ["consumo_nafta", "Numerica", "Consumo de combustible", "Feature con posible leakage"],
        ["id_bodega", "Categorica (ID)", "Identificador de bodega", "Feature categorica"],
        ["target_tiempo_entrega", "Numerica continua", "Tiempo final de entrega (minutos)", "Target continuo"],
    ],
    columns=["columna", "tipo", "descripcion", "rol_sugerido"],
)

display(column_roles)


## 6. Analisis de valores nulos


In [ ]:
missing_table = exploratory_analysis.missing_values_table(df)
display(missing_table)

if missing_table["cantidad_nulos"].sum() == 0:
    display(Markdown("**Conclusion:** No se detectan valores nulos en el dataset original. No se requiere imputacion inicial, pero se mantiene validacion en el pipeline por reproducibilidad."))


## 7. Analisis de duplicados


In [ ]:
duplicates_table = exploratory_analysis.duplicated_summary(df)
display(duplicates_table)

if int(duplicates_table.loc[0, "duplicados_exactos"]) == 0:
    display(Markdown("**Conclusion:** No se detectan duplicados exactos."))


## 8. Analisis de variables categoricas


In [ ]:
df_base = data_preprocessing.normalize_column_names(df)
df_base = data_preprocessing.clean_text_columns(df_base, data_preprocessing.TEXT_CATEGORICAL_COLUMNS)
df_base = data_preprocessing.convert_id_bodega_to_category(df_base)

categorical_columns = ["trafico_nivel", "clima", "tipo_vehiculo", "id_bodega"]
categorical_counts_table = exploratory_analysis.categorical_value_counts(df_base, categorical_columns)
display(categorical_counts_table)

for col in categorical_columns:
    if col in df_base.columns:
        print(f"Categorias en {col}:", sorted(df_base[col].astype(str).unique().tolist()))

plots_dir = PROJECT_ROOT / "results" / "plots"
categorical_plot_paths = visualization.plot_categorical_counts(df_base, categorical_columns, plots_dir)
print("Graficos categoricos guardados:")
for path in categorical_plot_paths:
    print("-", path)


Conclusiones esperadas de categoricas:
- `trafico_nivel` tiene orden logico: Bajo < Medio < Alto < Critico.
- `clima` y `tipo_vehiculo` no tienen orden natural.
- `id_bodega` debe tratarse como variable categorica, no continua.


## 9. Analisis de variables numericas


In [ ]:
numeric_columns = [
    "distancia_km",
    "peso_carga_kg",
    "paradas_previas",
    "experiencia_chofer_anios",
    "hora_despacho",
    "costo_envio",
    "consumo_nafta",
    "target_tiempo_entrega",
]

numeric_table = exploratory_analysis.numeric_summary(df_base, numeric_columns)
display(numeric_table)

hist_paths = visualization.plot_numeric_histograms(df_base, numeric_columns, plots_dir)
box_paths = visualization.plot_boxplots(df_base, numeric_columns, plots_dir)
print("Histogramas guardados:", len(hist_paths))
print("Boxplots guardados:", len(box_paths))


## 10. Validacion de reglas de negocio


In [ ]:
business_rules_table = exploratory_analysis.business_rules_report(df_base)
display(business_rules_table)

invalid_target_rows = df_base.loc[pd.to_numeric(df_base["target_tiempo_entrega"], errors="coerce") <= 0]

print("Registros con target <= 0:", len(invalid_target_rows))
display(
    invalid_target_rows[
        ["distancia_km", "trafico_nivel", "clima", "tipo_vehiculo", "target_tiempo_entrega"]
    ]
)


**Conclusion obligatoria:**
Los registros con `target_tiempo_entrega` menor o igual a cero se consideran invalidos, porque una entrega no puede tener duracion negativa o nula. Se recomienda eliminarlos del dataset limpio.


## 11. Analisis especifico del target


In [ ]:
target_table = exploratory_analysis.target_summary(df_base, "target_tiempo_entrega")
display(target_table)

df_flagged = data_preprocessing.add_suspicious_time_flag(df_base, threshold=10)

target_invalid_table = df_flagged.loc[pd.to_numeric(df_flagged["target_tiempo_entrega"], errors="coerce") <= 0]
target_suspicious_table = df_flagged.loc[pd.to_numeric(df_flagged["target_tiempo_entrega"], errors="coerce") < 10]

print("Registros con target <= 0:", len(target_invalid_table))
print("Registros con target < 10:", len(target_suspicious_table))

display(target_invalid_table[["distancia_km", "trafico_nivel", "target_tiempo_entrega"]])
display(target_suspicious_table[["distancia_km", "trafico_nivel", "target_tiempo_entrega", "tiempo_sospechoso"]].head(20))

target_distribution_path = visualization.plot_target_distribution(
    df_flagged,
    target_column="target_tiempo_entrega",
    output_path=plots_dir / "target_distribution.png",
)
suspicious_plot_path = visualization.plot_suspicious_times(
    df_flagged,
    output_path=plots_dir / "target_suspicious_times.png",
)
print("Grafico target:", target_distribution_path)
print("Grafico sospechosos:", suspicious_plot_path)


Criterio aplicado:
- `target <= 0`: invalido, se elimina del dataset limpio.
- `target < 10`: se marca como sospechoso, pero no se elimina automaticamente.


## 12. Deteccion de outliers con IQR


In [ ]:
outliers_iqr_table = exploratory_analysis.detect_outliers_iqr(df_base, numeric_columns)
display(outliers_iqr_table)

target_outlier_info = outliers_iqr_table.loc[outliers_iqr_table["columna"] == "target_tiempo_entrega"]
display(target_outlier_info)


Interpretacion:
En logistica, un tiempo alto puede ser un evento real y relevante; por eso los outliers se revisan y documentan antes de decidir si eliminarlos.


## 13. Correlacion con el target


In [ ]:
corr_matrix = df_base[numeric_columns].corr(numeric_only=True)
display(corr_matrix)

corr_target_table = exploratory_analysis.correlation_with_target(df_base[numeric_columns], "target_tiempo_entrega")
display(corr_target_table)

corr_heatmap_path = visualization.plot_correlation_heatmap(
    df_base[numeric_columns],
    output_path=plots_dir / "correlation_heatmap.png",
)
print("Heatmap guardado en:", corr_heatmap_path)


Interpretacion:
Una baja correlacion lineal no significa que no exista relacion, pero si sugiere que el target podria depender de interacciones no lineales, variables categoricas o variables que no estan presentes en el dataset.

Variables futuras sugeridas:
- comuna o zona de entrega
- coordenadas o distancia real por ruta
- dia de la semana
- fecha
- ventana horaria real
- densidad urbana
- prioridad del pedido
- estado del repartidor
- historial de congestion
- tipo de ruta
- cantidad de pedidos simultaneos


## 14. Analisis target vs categoricas


In [ ]:
target_by_category_paths = visualization.plot_target_by_category(
    df_flagged,
    target_column="target_tiempo_entrega",
    categorical_columns=categorical_columns,
    output_dir=plots_dir,
)
print("Graficos target vs categoricas guardados:")
for path in target_by_category_paths:
    print("-", path)


## 15. Analisis target vs numericas


In [ ]:
numeric_feature_columns = [col for col in numeric_columns if col != "target_tiempo_entrega"]

scatter_paths = visualization.plot_scatter_numeric_vs_target(
    df_flagged,
    numeric_columns=numeric_feature_columns,
    target_column="target_tiempo_entrega",
    output_dir=plots_dir,
)

print("Graficos scatter guardados:")
for path in scatter_paths:
    print("-", path)

# Comparacion exploratoria antes y despues de StandardScaler (sin modelado)
scaler = StandardScaler()
scaled_array = scaler.fit_transform(df_flagged[numeric_feature_columns])
scaled_df = pd.DataFrame(scaled_array, columns=numeric_feature_columns)

scaling_comparison = pd.DataFrame(
    {
        "feature": numeric_feature_columns,
        "media_original": df_flagged[numeric_feature_columns].mean().values,
        "std_original": df_flagged[numeric_feature_columns].std(ddof=0).values,
        "media_escalada": scaled_df.mean().values,
        "std_escalada": scaled_df.std(ddof=0).values,
    }
)

display(scaling_comparison)


Nota de escalado:
- KNN, K-Means, PCA o SVR suelen requerir escalado (ej. `StandardScaler`).
- Arboles de decision y Random Forest no lo requieren estrictamente, aunque puede mantenerse por orden en pipelines.


## 16. Feature engineering exploratorio


In [ ]:
df_exploratory_features = feature_engineering.prepare_exploratory_features(df_flagged)

new_feature_columns = [
    "hora_sin",
    "hora_cos",
    "costo_por_km",
    "consumo_por_km",
    "carga_por_km",
    "paradas_por_km",
]

display(df_exploratory_features[new_feature_columns].head())
display(df_exploratory_features[new_feature_columns].describe().T)

feature_corr_table = exploratory_analysis.correlation_with_target(
    df_exploratory_features[numeric_columns + new_feature_columns],
    target_column="target_tiempo_entrega",
)
display(feature_corr_table)

display(feature_engineering.get_feature_recommendations())


## 17. Revision de posible leakage


La variable `consumo_nafta` puede tener dos interpretaciones:

- Si representa consumo estimado antes del despacho, puede utilizarse como predictor.
- Si representa consumo real observado despues de completar la entrega, seria fuga de informacion, porque no estaria disponible al momento de predecir.

Por lo tanto, en modelado posterior se recomienda comparar dos experimentos:
1. Modelo con `consumo_nafta`.
2. Modelo sin `consumo_nafta`.


## 18. Dataset limpio preliminar


In [ ]:
df_clean, cleaning_summary = data_preprocessing.clean_logistics_dataset(df, remove_invalid_target=True)
df_clean_features = feature_engineering.prepare_exploratory_features(df_clean)

print("Resumen de limpieza:")
for key, value in cleaning_summary.items():
    if key in {"business_rules_report", "invalid_target_rows"}:
        continue
    print(f"- {key}: {value}")

if not cleaning_summary["invalid_target_rows"].empty:
    print("
Registros removidos por target invalido:")
    display(cleaning_summary["invalid_target_rows"][["distancia_km", "trafico_nivel", "target_tiempo_entrega"]])

processed_clean_path = PROJECT_ROOT / "data" / "processed" / "logistica_limpio_eda.csv"
processed_features_path = PROJECT_ROOT / "data" / "processed" / "logistica_limpio_eda_features.csv"

df_clean.to_csv(processed_clean_path, index=False)
df_clean_features.to_csv(processed_features_path, index=False)

print("Dataset limpio guardado en:", processed_clean_path)
print("Dataset limpio + features guardado en:", processed_features_path)


## 19. Exportacion de reportes


In [ ]:
reports_dir = PROJECT_ROOT / "results" / "reports"
metrics_dir = PROJECT_ROOT / "results" / "metrics"
reports_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

overview_table.to_csv(reports_dir / "eda_dataset_overview.csv", index=False)
missing_table.to_csv(reports_dir / "eda_missing_values.csv", index=False)
duplicates_table.to_csv(reports_dir / "eda_duplicates.csv", index=False)
categorical_counts_table.to_csv(reports_dir / "eda_categorical_counts.csv", index=False)
numeric_table.to_csv(reports_dir / "eda_numeric_summary.csv", index=False)
business_rules_table.to_csv(reports_dir / "eda_business_rules_report.csv", index=False)
outliers_iqr_table.to_csv(reports_dir / "eda_outliers_iqr.csv", index=False)
corr_target_table.to_csv(reports_dir / "eda_correlations_target.csv", index=False)
target_suspicious_table.to_csv(reports_dir / "eda_suspicious_times.csv", index=False)
scaling_comparison.to_csv(metrics_dir / "eda_scaling_comparison.csv", index=False)

for category_col in categorical_columns:
    category_df = categorical_counts_table[categorical_counts_table["columna"] == category_col]
    category_df.to_csv(reports_dir / f"eda_categorical_counts_{category_col}.csv", index=False)

print("Reportes guardados en:", reports_dir)
print("Metricas guardadas en:", metrics_dir)


## 20. Conclusiones del EDA


1. El dataset presenta buena estructura general para analisis y modelado.
2. No se detectan valores nulos en el dataset original.
3. No se detectan duplicados exactos.
4. Las variables categoricas principales estan consistentes tras limpieza de texto.
5. `id_bodega` debe tratarse como categorica, no como variable continua.
6. El principal problema de limpieza esta en `target_tiempo_entrega` por registros negativos/imposibles.
7. Los tiempos menores a 10 minutos se marcan como sospechosos y no se eliminan automaticamente.
8. Los outliers del target deben revisarse con criterio de negocio antes de cualquier descarte.
9. `consumo_nafta` puede representar leakage dependiendo de su definicion operacional real.
10. Las correlaciones lineales con el target son bajas; esto no descarta relaciones no lineales o interacciones.
11. Se recomiendan features adicionales: `costo_por_km`, `consumo_por_km`, `carga_por_km`, `paradas_por_km`, `hora_sin` y `hora_cos`.
12. El dataset queda preparado para modelado supervisado y no supervisado en notebooks posteriores.


## 21. Generar docs/eda_hallazgos.md


In [ ]:
rows_count = int(df.shape[0])
cols_count = int(df.shape[1])
num_count = int(len(df.select_dtypes(include=[np.number]).columns))
cat_count = int(len(df.select_dtypes(exclude=[np.number]).columns))

invalid_count = int((pd.to_numeric(df_base["target_tiempo_entrega"], errors="coerce") <= 0).sum())
suspicious_count = int((pd.to_numeric(df_base["target_tiempo_entrega"], errors="coerce") < 10).sum())

eda_md = f'''# Hallazgos del Analisis Exploratorio de Datos

## Contexto
El proyecto analiza datos de Logistica 4.0 (Last Mile) para comprender factores asociados al tiempo de entrega y preparar el dataset para modelado posterior.

## Estructura del dataset
- Filas: {rows_count}
- Columnas: {cols_count}
- Target principal: `target_tiempo_entrega` (continua)
- Variables numericas detectadas inicialmente: {num_count}
- Variables categoricas detectadas inicialmente: {cat_count}

## Calidad de datos
- Valores nulos: 0
- Duplicados exactos: 0
- Categorias revisadas y normalizadas en trafico, clima y tipo de vehiculo.
- Reglas de negocio validadas para rangos operacionales.

## Problemas detectados
- Registros con target invalido (`<= 0`): {invalid_count}
- Registros con tiempo sospechoso (`< 10` minutos): {suspicious_count}
- Outliers detectados mediante IQR, incluyendo outliers en el target.

## Decisiones de limpieza
- Se eliminan registros con `target_tiempo_entrega <= 0` para dataset limpio.
- Los registros con `target_tiempo_entrega < 10` se conservan y se marcan con `tiempo_sospechoso`.
- `id_bodega` se convierte a categoria.
- No se eliminan outliers automaticamente sin validacion de negocio.

## Variables categoricas y encoding recomendado
- `clima`: One-Hot Encoding.
- `tipo_vehiculo`: One-Hot Encoding.
- `id_bodega`: One-Hot Encoding o tratamiento categorico.
- `trafico_nivel`: comparar encoding ordinal (Bajo=0, Medio=1, Alto=2, Critico=3) versus One-Hot.

## Posible fuga de informacion
`consumo_nafta` puede ser leakage si corresponde a consumo real posterior a la entrega. Se recomienda validar definicion y comparar modelos con/sin esta variable.

## Recomendaciones para modelado posterior
- Modelos supervisados de regresion para `target_tiempo_entrega`.
- Clasificacion derivada (por ejemplo, `entrega_tardia` o `tiempo_alto`) en experimentos separados.
- Clustering para patrones logisticos.
- Escalado con `StandardScaler` para metodos sensibles a distancia.
- Comparar experimentos con y sin `consumo_nafta`.

## Conclusion
El dataset queda preparado para la siguiente etapa de modelado, con una limpieza trazable, riesgos documentados y features exploratorias recomendadas.
'''

eda_doc_path = PROJECT_ROOT / "docs" / "eda_hallazgos.md"
eda_doc_path.write_text(eda_md, encoding="utf-8")
print("Documento generado en:", eda_doc_path)
